# Text Representations — Exercises

Companion to the note [Text Representations](Text%20Representations.md).

Practise turning text into vectors: bag-of-words, TF-IDF, cosine similarity, word2vec (skip-gram with negative sampling), count-based PPMI + SVD embeddings and fastText subwords, by hand and from scratch in NumPy, checked against scikit-learn where it has a reference implementation.

**17 exercises** · 🧠 Concept ×4 · ✍️ By hand ×6 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
rng = np.random.default_rng(0)

docs = ["the cat sat on the mat",
        "the dog sat on the log",
        "cats and dogs are pets",
        "the stock market fell sharply today",
        "markets rallied as stocks rose"]

# Toy corpus for word embeddings: animals and vehicles appear in parallel contexts
emb_corpus = ["the cat drinks milk", "the dog drinks milk", "a cat eats fish", "a dog eats meat",
              "the car needs fuel", "the truck needs fuel", "a car burns petrol", "a truck burns diesel",
              "the cat chases the dog", "the truck passes the car"]
emb_sents = [s.split() for s in emb_corpus]
emb_vocab = sorted({w for s in emb_sents for w in s})
w2i = {w: i for i, w in enumerate(emb_vocab)}

## Part A · Concepts

### Exercise 1 · What does bag-of-words throw away?
*🧠 Concept · ★☆☆*

"dog bites man" and "man bites dog" get identical bag-of-words vectors. List three kinds of information BoW loses, and for
each name a representation from the note's table that (partly) recovers it.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Think about word order, meaning similarity between different words, and words with several meanings.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. **Order**: recovered locally by **n-grams** ("dog bites" ≠ "man bites"), fully by contextual models.
2. **Similarity between words**: "film" and "movie" are orthogonal one-hot dimensions; **dense embeddings** (word2vec, GloVe)
place them close together.
3. **Word sense**: "bank" (river vs money) gets one dimension/vector; **contextual embeddings** (ELMo, BERT) give a different
vector in each sentence.
Also lost: morphology ("run"/"running" are unrelated features, recovered by **fastText** subwords) and very high
dimensionality / sparsity.

</details>

### Exercise 2 · Why the IDF factor?
*🧠 Concept · ★☆☆*

With $\text{tfidf}(t,d) = tf(t,d)\cdot\log\frac{N}{df(t)}$: what weight does a word that occurs in **every** document get?
Why is that sensible for search and classification, and in which situation could it hurt?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Compute $\log(N/N)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$df = N \Rightarrow \log 1 = 0$: the word is ignored completely. Words that appear everywhere ("the", "is") carry no
information about *which* document is relevant, so down-weighting them makes rare, specific terms dominate the
similarity. It can hurt when a frequent word *is* informative for the task, e.g. "not" in sentiment, or in a narrow corpus
where every document mentions the key term (e.g. all are about "covid"). sklearn's smoothed variant $\ln\frac{1+N}{1+df}+1$ never
reaches exactly 0 for that reason (and to avoid division by zero).

</details>

### Exercise 3 · Static vs contextual embeddings
*🧠 Concept · ★★☆*

Consider "I sat by the river **bank**" and "I opened a **bank** account". What does word2vec return for "bank" in the two
sentences, and what does BERT return? Why are sentence-embedding models (SBERT, E5, BGE) trained with a **contrastive** loss
rather than simply averaging BERT token vectors?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

word2vec is a lookup table. For the last part, see [[Self-Supervised and Contrastive Learning]].

</details>

<details>
<summary><b>✅ Solution</b></summary>

word2vec is a lookup table: the *same* vector in both sentences, a blend of both senses (dominated by the more frequent one).
BERT computes the vector from the whole sentence with self-attention, so the two "bank" vectors differ and cluster by sense.
Averaging raw BERT token vectors gives poor sentence similarity: the embedding space is anisotropic (all vectors point in
similar directions) and was never trained so that *cosine* reflects *meaning*. Contrastive training (pull paraphrase / query–passage
pairs together, push in-batch negatives apart) directly optimises cosine similarity for semantic search and [[RAG and Agents]].

</details>

### Exercise 4 · Out-of-vocabulary words
*🧠 Concept · ★★☆*

A word2vec model trained on news meets the words "unfollowable" and "recieve" (typo) at test time. What happens with
word2vec, with fastText, and with a BPE-tokenised transformer?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

fastText represents a word as the sum of its character n-gram vectors.

</details>

<details>
<summary><b>✅ Solution</b></summary>

**word2vec**: both are OOV → no vector (map to `<UNK>` or drop), losing the information.
**fastText**: the vector is the sum of character n-gram vectors (`<un`, `unf`, …, `ble>`), so it composes a sensible vector
from known pieces ("un-", "follow", "-able") and the typo shares most n-grams with "receive".
**BPE transformer**: the word is split into known subwords (e.g. `un` `follow` `able`) and the model contextualises them;
there is never an OOV, at the cost of longer sequences.

</details>

## Part B · By hand

### Exercise 5 · TF-IDF by hand
*✍️ By hand · ★☆☆*

Corpus of $N=3$ documents: $d_1$ = "cat sat mat", $d_2$ = "cat cat dog", $d_3$ = "dog ran". Use raw counts for $tf$ and
the note's $\text{idf}(t) = \ln\frac{N}{df(t)}$ (natural log). Compute $\text{tfidf}(\text{cat}, d_2)$ and
$\text{tfidf}(\text{sat}, d_1)$. Which term in $d_2$ has the larger weight?

In [ ]:
tfidf_cat_d2 = None
tfidf_sat_d1 = None

check('tfidf(cat, d2)', tfidf_cat_d2, 2*np.log(3/2))
check('tfidf(sat, d1)', tfidf_sat_d1, np.log(3))

<details>
<summary><b>💡 Hint</b></summary>

$df(\text{cat}) = 2$, $df(\text{sat}) = 1$, $df(\text{dog}) = 2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\text{tfidf}(\text{cat}, d_2) = 2\ln\tfrac32 \approx 0.811$; $\text{tfidf}(\text{sat}, d_1) = 1\cdot\ln 3 \approx 1.099$.
In $d_2$, "dog" gets $1\cdot\ln\frac32\approx 0.405$, so "cat" (counted twice) has the larger weight.
A word that occurs once in only one document ("sat") can outweigh a repeated but more common word.

```python
tfidf_cat_d2 = 2 * np.log(1.5)
tfidf_sat_d1 = np.log(3)
```

</details>

### Exercise 6 · Cosine similarity by hand
*✍️ By hand · ★☆☆*

Compute $\cos(a,b) = \frac{a^\top b}{\lVert a\rVert\,\lVert b\rVert}$ for $a = (1,2,0)$, $b = (2,1,1)$.
Why is cosine preferred to Euclidean distance for comparing TF-IDF vectors of documents of very different lengths?

In [ ]:
cos_ab = None

check('cos(a,b)', cos_ab, cosine_similarity([[1,2,0]], [[2,1,1]])[0,0])

<details>
<summary><b>💡 Hint</b></summary>

$a^\top b = 4$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$a^\top b = 2+2+0 = 4$, $\lVert a\rVert = \sqrt5$, $\lVert b\rVert=\sqrt6$, so $\cos = 4/\sqrt{30} \approx 0.730$.
A document repeated twice has a count vector twice as long: Euclidean distance changes, the *direction* (and hence cosine)
does not. Cosine compares the **topic mix**, not the length. For unit-normalised vectors, $\lVert a-b\rVert^2 = 2 - 2\cos(a,b)$, so
the two rankings coincide.

```python
cos_ab = 4 / np.sqrt(30)
```

</details>

### Exercise 7 · Word analogies by vector arithmetic
*✍️ By hand · ★☆☆*

Toy 2-D embeddings: man $(1,0)$, woman $(1,2)$, king $(4,0.5)$, queen $(4.2,2.4)$, prince $(3.5,0.3)$, apple $(-1,3)$.
Compute $v = \text{king} - \text{man} + \text{woman}$ and find the nearest word by cosine similarity, **excluding** the three
query words (as word2vec's analogy evaluation does). Store the word as a string.

In [ ]:
toy_emb = {'man': (1, 0), 'woman': (1, 2), 'king': (4, 0.5), 'queen': (4.2, 2.4), 'prince': (3.5, 0.3), 'apple': (-1, 3)}
analogy_answer = None  # e.g. 'prince'

_E = {k: np.array(v, float) for k, v in toy_emb.items()}
_v = _E['king'] - _E['man'] + _E['woman']
_c = {w: _v @ e / np.linalg.norm(_v) / np.linalg.norm(e) for w, e in _E.items() if w not in ('king', 'man', 'woman')}
check('analogy', analogy_answer, max(_c, key=_c.get))

<details>
<summary><b>💡 Hint</b></summary>

$v = (4, 2.5)$. Compare its cosine with queen, prince and apple.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$v = (4-1+1,\ 0.5-0+2) = (4, 2.5)$. Cosines: queen $\approx 0.999$, prince $\approx 0.89$, apple $\approx 0.23$ → **queen**.
The "gender" direction woman − man $= (0, 2)$ is (approximately) constant across pairs, which is what makes the parallelogram work.
Without excluding the query words, the nearest neighbour is often one of the inputs (here woman scores $\approx 0.85$, still below queen).

```python
toy_emb = {'man': (1, 0), 'woman': (1, 2), 'king': (4, 0.5), 'queen': (4.2, 2.4), 'prince': (3.5, 0.3), 'apple': (-1, 3)}
analogy_answer = 'queen'
```

</details>

### Exercise 8 · How many skip-gram training pairs?
*✍️ By hand · ★☆☆*

Skip-gram predicts each context word within a window of $\pm m$ positions from the centre word. For the sentence
"the quick brown fox jumps" (5 tokens) and $m=2$, how many (centre, context) pairs are generated? List the pairs with centre "brown".

In [ ]:
n_pairs = None

check('number of pairs', n_pairs, sum(min(4, i + 2) - max(0, i - 2) for i in range(5)))

<details>
<summary><b>💡 Hint</b></summary>

Count the neighbours of each position: the ends have fewer.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Neighbours per position: 2, 3, 4, 3, 2 → **14** pairs. Centre "brown": (brown, the), (brown, quick), (brown, fox), (brown, jumps).
In general $\approx 2mT$ pairs for $T$ tokens, so a 1B-token corpus with $m=5$ yields $\sim 10^{10}$ updates, which is why
negative sampling (cheap per-update cost) matters.

```python
n_pairs = 14
```

</details>

### Exercise 9 · The negative-sampling gradient
*✍️ By hand · ★★★*

Skip-gram with negative sampling minimises, for centre vector $v_c$, true context vector $u_o$ and negatives $u_1,\dots,u_K$,
$$L = -\log\sigma(u_o^\top v_c) - \sum_{k=1}^K \log\sigma(-u_k^\top v_c).$$
1. Derive $\nabla_{v_c} L$ and $\nabla_{u_o} L$.
2. Evaluate $\nabla_{v_c}L$ for $v_c = (1,0)$, $u_o = (1,1)$ and one negative $u_1 = (0,1)$.

In [ ]:
grad_vc = None  # [g1, g2]

def _L(v):
    s = lambda z: 1 / (1 + np.exp(-z))
    return -np.log(s(np.array([1., 1.]) @ v)) - np.log(s(-np.array([0., 1.]) @ v))
_v0, _h = np.array([1., 0.]), 1e-6
_num = [(_L(_v0 + _h * e) - _L(_v0 - _h * e)) / (2 * _h) for e in np.eye(2)]
check('gradient wrt v_c (vs finite differences)', grad_vc, _num, tol=1e-5)

<details>
<summary><b>💡 Hint 1</b></summary>

Use $\frac{d}{dz}\log\sigma(z) = 1 - \sigma(z) = \sigma(-z)$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$u_o^\top v_c = 1$ and $u_1^\top v_c = 0$, so you need $\sigma(1) \approx 0.731$ and $\sigma(0) = 0.5$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $\nabla_{v_c} L = (\sigma(u_o^\top v_c) - 1)\,u_o + \sum_k \sigma(u_k^\top v_c)\,u_k$ and
$\nabla_{u_o} L = (\sigma(u_o^\top v_c) - 1)\,v_c$, $\nabla_{u_k} L = \sigma(u_k^\top v_c)\,v_c$.
Read it as "(prediction − label) × other vector", exactly like [[Logistic Regression]] with label 1 for the true context and 0 for negatives.
2. $(\sigma(1)-1)(1,1) + \sigma(0)(0,1) = (-0.2689, -0.2689) + (0, 0.5) = (-0.2689,\ 0.2311)$.
Gradient descent moves $v_c$ towards $u_o$ and away from $u_1$.

```python
_s1 = 1 / (1 + np.exp(-1.0))
grad_vc = [_s1 - 1, (_s1 - 1) + 0.5]
```

</details>

### Exercise 10 · The 3/4-power noise distribution
*✍️ By hand · ★★☆*

word2vec draws negatives from $P_n(w) \propto c(w)^{3/4}$. For three words with counts 100, 10 and 1, compute the probability
of drawing the **rare** word under (a) the raw unigram distribution and (b) $P_n$. What is the effect of the exponent?

In [ ]:
p_rare_unigram = None
p_rare_noise = None

_c = np.array([100, 10, 1.])
check('unigram', p_rare_unigram, _c[2] / _c.sum())
check('3/4 power', p_rare_noise, _c[2]**0.75 / (_c**0.75).sum())

<details>
<summary><b>💡 Hint</b></summary>

$100^{0.75} = 10^{1.5} \approx 31.62$ and $10^{0.75} \approx 5.62$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $1/111 \approx 0.0090$. (b) $1/(31.62 + 5.62 + 1) = 1/38.25 \approx 0.0261$, almost 3× more often.
The exponent flattens the distribution: frequent words are sampled less and rare words more, so rare words get more negative
updates (and frequent ones like "the" do not dominate). It was chosen empirically by Mikolov et al.

```python
p_rare_unigram = 1 / 111
p_rare_noise = 1 / (100**0.75 + 10**0.75 + 1)
```

</details>

## Part C · Code from scratch

### Exercise 11 · Bag-of-words from scratch
*💻 Code · ★☆☆*

Implement `bow(docs)` returning `(vocab, X)` where `vocab` is the sorted list of tokens (split on whitespace) and
`X[i, j]` counts token `vocab[j]` in document `i`. Compare with `CountVectorizer`.

In [ ]:
def bow(docs):
    # TODO
    return None

bow_out = bow(docs)

_cv = CountVectorizer(token_pattern=r"\S+")
_Xr = _cv.fit_transform(docs).toarray()
check('vocabulary', None if bow_out is None else list(bow_out[0]) == list(_cv.get_feature_names_out()), True)
check('count matrix', None if bow_out is None else bow_out[1], _Xr)

<details>
<summary><b>💡 Hint</b></summary>

Build the vocabulary with `sorted({w for d in docs for w in d.split()})`, then a `{word: column}` dict.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The matrix is 5 × 22 and about three quarters zeros (26 non-zeros out of 110) even for this tiny corpus; real vocabularies have $10^5$ columns, which is why
sklearn returns a `scipy.sparse` matrix.

```python
def bow(docs):
    vocab = sorted({w for d in docs for w in d.split()})
    col = {w: j for j, w in enumerate(vocab)}
    X = np.zeros((len(docs), len(vocab)), dtype=int)
    for i, d in enumerate(docs):
        for w in d.split():
            X[i, col[w]] += 1
    return vocab, X

bow_out = bow(docs)
```

</details>

### Exercise 12 · TF-IDF exactly like scikit-learn
*💻 Code · ★★☆*

scikit-learn's `TfidfVectorizer` (defaults) differs from the note's formula in two ways: the IDF is **smoothed**,
$\text{idf}(t) = \ln\frac{1+N}{1+df(t)} + 1$, and every row is **L2-normalised**. Implement `tfidf(X_counts)` and match
`TfidfVectorizer(token_pattern=r"\S+")` on `docs`.

In [ ]:
def tfidf(Xc):
    # TODO: Xc is a dense (n_docs, n_terms) count matrix
    return None

Xc = CountVectorizer(token_pattern=r"\S+").fit_transform(docs).toarray()
X_tfidf = tfidf(Xc)

check('tf-idf matrix', X_tfidf, TfidfVectorizer(token_pattern=r'\S+').fit_transform(docs).toarray())

<details>
<summary><b>💡 Hint</b></summary>

`df = (Xc > 0).sum(0)`; multiply each column by its idf, then divide each row by its norm.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The "+1" keeps terms that occur in every document (here none, but "the" is close) from vanishing, and the "1+" in numerator and
denominator acts like one extra document containing every term (no division by zero). L2 normalisation makes the dot product
equal to cosine similarity.

```python
def tfidf(Xc):
    N = Xc.shape[0]
    df = (Xc > 0).sum(0)
    idf = np.log((1 + N) / (1 + df)) + 1
    W = Xc * idf
    return W / np.linalg.norm(W, axis=1, keepdims=True)

Xc = CountVectorizer(token_pattern=r"\S+").fit_transform(docs).toarray()
X_tfidf = tfidf(Xc)
```

</details>

### Exercise 13 · A tiny TF-IDF search engine
*💻 Code · ★☆☆*

Fit a `TfidfVectorizer` on `docs`, embed the query "stocks fell in the market" and rank documents by cosine similarity
(compute the cosines yourself with NumPy). Store the indices sorted best-first in `ranking`. Which document wins, and
why does document 4 ("markets rallied as stocks rose") not score higher?

In [ ]:
vec = TfidfVectorizer().fit(docs)
query = "stocks fell in the market"
ranking = None  # e.g. np.array([3, 4, 0, 1, 2])

_s = cosine_similarity(vec.transform([query]), vec.transform(docs))[0]
check('ranking', ranking, np.argsort(-_s, kind='stable'))

<details>
<summary><b>💡 Hint</b></summary>

Rows of `vec.transform(...)` are already L2-normalised, so cosine = dot product: `(D @ q.T)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Document 3 ("the stock market fell sharply today") wins via the exact words "fell" and "market". Document 4 is about the same
topic, but "stocks" ≠ "stock" and "markets" ≠ "market" as tokens, so it only matches "stocks". This vocabulary mismatch is
exactly what dense embeddings (and stemming, to a lesser degree) fix; in RAG, **hybrid** retrieval combines both ([[RAG and Agents]]).

```python
vec = TfidfVectorizer().fit(docs)
query = "stocks fell in the market"
D = vec.transform(docs).toarray()
q = vec.transform([query]).toarray()[0]
scores = D @ q
ranking = np.argsort(-scores, kind='stable')
print(np.round(scores, 3), ranking)
```

</details>

### Exercise 14 · fastText character n-grams
*💻 Code · ★☆☆*

Implement `char_ngrams(word, n)` that wraps the word in boundary markers `<` and `>` and returns all character n-grams
in order. For "where" and $n=3$ you should get the example from the fastText paper. Then compute the **Jaccard overlap** of the
3-gram sets of "receive" and the typo "recieve".

In [ ]:
def char_ngrams(word, n=3):
    # TODO
    return None

jacc_typo = None  # |A ∩ B| / |A ∪ B| for the 3-gram sets of 'receive' and 'recieve'

_ng = char_ngrams('where', 3)
check('where', None if _ng is None else _ng == ['<wh', 'whe', 'her', 'ere', 're>'], True)
_A = {'<re','rec','ece','cei','eiv','ive','ve>'}; _B = {'<re','rec','eci','cie','iev','eve','ve>'}
check('Jaccard', jacc_typo, len(_A & _B) / len(_A | _B))

<details>
<summary><b>💡 Hint</b></summary>

`w = '<' + word + '>'`, then `[w[i:i+n] for i in range(len(w) - n + 1)]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

"where" → `<wh, whe, her, ere, re>` (note `her` is different from the word `<her>`, thanks to the markers).
"receive" and "recieve" share `<re, rec, ve>` → Jaccard $3/11 \approx 0.27$. fastText actually uses all $n$ from 3 to 6, which
gives more shared pieces; the word vector is the sum of the n-gram vectors, so the typo lands near the correct word.

```python
def char_ngrams(word, n=3):
    w = '<' + word + '>'
    return [w[i:i + n] for i in range(len(w) - n + 1)]

A, B = set(char_ngrams('receive')), set(char_ngrams('recieve'))
jacc_typo = len(A & B) / len(A | B)
```

</details>

### Exercise 15 · Count-based embeddings: PPMI + SVD
*💻 Code · ★★☆*

On `emb_corpus` build the word–context co-occurrence matrix $C$ with a window of $\pm2$, convert it to **positive PMI**
$$\text{PPMI}(w,c) = \max\Big(0, \log\frac{P(w,c)}{P(w)P(c)}\Big),$$
and take a rank-5 truncated SVD, $E = U_{:,:5}\,\Sigma_{:5}$, as word embeddings (this is the GloVe/LSA family of the note).
Store `cos_same` = mean of cos(cat, dog) and cos(car, truck), `cos_diff` = mean of cos(cat, truck) and cos(dog, car).

In [ ]:
def cooc_matrix(sents, window=2):
    # TODO: (V, V) counts using w2i
    return None

cos_same = None
cos_diff = None

if cos_same is not None:
    check('similar words are closer than unrelated ones', cos_same > cos_diff + 0.3, True)
else:
    check('similar words are closer than unrelated ones', None, True)

<details>
<summary><b>💡 Hint 1</b></summary>

$P(w,c) = C_{wc}/\sum C$, $P(w)$ = row sum / total, $P(c)$ = column sum / total. Use `np.errstate(divide='ignore')` for $\log 0$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

`U, S, Vt = np.linalg.svd(ppmi)`; `E = U[:, :5] * S[:5]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

"cat" and "dog" occur with the same contexts (drinks, milk, eats, the, a) and get cosine ≈ 1, "car"/"truck" likewise, while
cross-group pairs are near 0.1. Levy & Goldberg (2014) showed that skip-gram with negative sampling implicitly factorises a
*shifted* PMI matrix, so this count-based method and word2vec are two routes to the same idea: **distributional similarity**.

```python
def cooc_matrix(sents, window=2):
    C = np.zeros((len(w2i), len(w2i)))
    for s in sents:
        for i, w in enumerate(s):
            for j in range(max(0, i - window), min(len(s), i + window + 1)):
                if j != i:
                    C[w2i[w], w2i[s[j]]] += 1
    return C

C = cooc_matrix(emb_sents)
tot = C.sum()
pw, pc = C.sum(1) / tot, C.sum(0) / tot
with np.errstate(divide='ignore'):
    ppmi = np.maximum(np.log((C / tot) / np.outer(pw, pc)), 0)
U, S, Vt = np.linalg.svd(ppmi)
E = U[:, :5] * S[:5]
cs = lambda a, b: E[w2i[a]] @ E[w2i[b]] / np.linalg.norm(E[w2i[a]]) / np.linalg.norm(E[w2i[b]])
cos_same = (cs('cat', 'dog') + cs('car', 'truck')) / 2
cos_diff = (cs('cat', 'truck') + cs('dog', 'car')) / 2
print(round(cos_same, 3), round(cos_diff, 3))
```

</details>

### Exercise 16 · word2vec: skip-gram with negative sampling
*💻 Code · ★★★*

Train skip-gram with negative sampling on `emb_corpus` with SGD, using your gradients from the by-hand exercise:

- generate (centre, context) pairs with window 2; noise distribution $\propto c(w)^{3/4}$ over context counts;
- input vectors `W_in` (init $\mathcal N(0, 0.1^2)$, dim 10) and output vectors `W_out` (init 0); $K=3$ negatives, lr 0.05, 200 epochs;
- per pair: $g = \sigma(U v_c) - \text{labels}$ for the rows $U$ = [true context; negatives], update `W_out` rows by $g\,v_c$ and $v_c$ by $g^\top U$.

Store `w2v_same` and `w2v_diff` as in the PPMI exercise (with `W_in` as embeddings).

In [ ]:
def skipgram_pairs(sents, window=2):
    # TODO: np.array of (centre_id, context_id)
    return None

def train_sgns(pairs, V, dim=10, K=3, lr=0.05, epochs=200, seed=0):
    # TODO: return W_in
    return None

w2v_same = None
w2v_diff = None

if w2v_same is not None:
    check('animals and vehicles separate', w2v_same > w2v_diff + 0.3, True)
else:
    check('animals and vehicles separate', None, True)

<details>
<summary><b>💡 Hint 1</b></summary>

Sample negatives with `r.choice(V, size=K, p=noise)`; occasionally a negative equals the true context, which is fine for a toy.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Compute `gv = g @ W_out[ids]` **before** updating `W_out`, then update both.

</details>

<details>
<summary><b>✅ Solution</b></summary>

After training, cos(cat, dog) and cos(car, truck) are around 0.85, while the cross pairs are around 0.1–0.15: the model has
discovered two semantic groups from co-occurrence alone. Real word2vec adds subsampling of frequent words, a decaying learning rate
and vocabularies of millions, but the update is exactly this one.

```python
def skipgram_pairs(sents, window=2):
    P = []
    for s in sents:
        for i, w in enumerate(s):
            for j in range(max(0, i - window), min(len(s), i + window + 1)):
                if j != i:
                    P.append((w2i[w], w2i[s[j]]))
    return np.array(P)

def train_sgns(pairs, V, dim=10, K=3, lr=0.05, epochs=200, seed=0):
    r = np.random.default_rng(seed)
    cnt = np.bincount(pairs[:, 1], minlength=V).astype(float)
    noise = cnt ** 0.75 / (cnt ** 0.75).sum()
    W_in, W_out = r.normal(0, 0.1, (V, dim)), np.zeros((V, dim))
    labels = np.r_[1.0, np.zeros(K)]
    for _ in range(epochs):
        for c, o in pairs[r.permutation(len(pairs))]:
            ids = np.r_[o, r.choice(V, size=K, p=noise)]
            v = W_in[c]
            g = 1 / (1 + np.exp(-(W_out[ids] @ v))) - labels
            gv = g @ W_out[ids]
            W_out[ids] -= lr * g[:, None] * v[None, :]
            W_in[c] -= lr * gv
    return W_in

W_in = train_sgns(skipgram_pairs(emb_sents), len(w2i))
cw = lambda a, b: W_in[w2i[a]] @ W_in[w2i[b]] / np.linalg.norm(W_in[w2i[a]]) / np.linalg.norm(W_in[w2i[b]])
w2v_same = (cw('cat', 'dog') + cw('car', 'truck')) / 2
w2v_diff = (cw('cat', 'truck') + cw('dog', 'car')) / 2
print(round(w2v_same, 3), round(w2v_diff, 3))
```

</details>

### Exercise 17 · Sentence embeddings by mean pooling
*💻 Code · ★★☆*

A cheap sentence embedding is the average of its word vectors. Using the PPMI embeddings `E` from the PPMI + SVD exercise (recompute them if needed),
embed "the dog drinks milk", "a cat eats fish" and "a truck burns diesel", and store the 3×3 cosine-similarity matrix in
`sent_sim`. Then explain two weaknesses of mean pooling that contrastively trained sentence encoders fix.

In [ ]:
def sentence_vec(sentence, E):
    # TODO: mean of the rows of E for the words in the sentence
    return None

test_sents = ["the dog drinks milk", "a cat eats fish", "a truck burns diesel"]
sent_sim = None

if sent_sim is not None:
    _S = np.asarray(sent_sim)
    check('animal sentences closer to each other than to the truck', _S[0, 1] > max(_S[0, 2], _S[1, 2]), True)
    check('symmetric with unit diagonal', np.allclose(_S, _S.T) and np.allclose(np.diag(_S), 1), True)
else:
    check('animal sentences closer to each other than to the truck', None, True)

<details>
<summary><b>💡 Hint</b></summary>

If `E` is not defined (you skipped the PPMI exercise), rebuild it from `cooc_matrix` first. `cosine_similarity` does the matrix in one call.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The two animal sentences are most similar even though they share no content word, because their words live in the same
region of the embedding space. Weaknesses of mean pooling: (1) **order and negation are lost** ("dog bites man" = "man bites dog",
"not good" ≈ "good"); (2) **frequent function words dominate** the average (fix: TF-IDF-weighted averaging or removing the top
principal component, as in SIF). Contrastive encoders (SBERT, E5, BGE) are trained end-to-end so that cosine reflects meaning.

```python
def sentence_vec(sentence, E):
    return E[[w2i[w] for w in sentence.split()]].mean(0)

C = cooc_matrix(emb_sents); tot = C.sum()
with np.errstate(divide='ignore'):
    ppmi = np.maximum(np.log((C / tot) / np.outer(C.sum(1) / tot, C.sum(0) / tot)), 0)
U, S, Vt = np.linalg.svd(ppmi); E = U[:, :5] * S[:5]
test_sents = ["the dog drinks milk", "a cat eats fish", "a truck burns diesel"]
sent_sim = cosine_similarity(np.array([sentence_vec(s, E) for s in test_sents]))
print(np.round(sent_sim, 2))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Text Representations](Text%20Representations.md).*